# Unlearning Precision Metrics

This notebook visualizes **precision metrics** for unlearning experiments: how well do weight changes from each unlearning method align with the ground-truth mask of parameters that were trained on the target data?

**Metrics computed by `scripts/precision_metrics.py`:**
1. **ROC AUC** — using multiple scoring modes:
   - **Raw |Δw|** — absolute weight change (global ranking)
   - **Quantile** — within-component percentile rank (normalizes across parameter scales)
   - **CompNorm** — |Δw| normalized by UnMask baseline mean delta per parameter
   - **Contrast** — |Δw_mask| − |Δw_unmask| (difference vs UnMask baseline)
   - **ContrastNorm** — symmetric contrast index (|Δ_mask| − |Δ_unmask|) / (|Δ_mask| + |Δ_unmask|), bounded [-1, 1]
   - **ContrastLN** — contrast score normalized by std within (layer, component)
   - **SignRev** — sign-reversal of injection direction: −(injection_delta × unlearn_delta)
   - **LayerNorm** — |Δw| / std(|Δw|) within same (layer, component) group
   - **Reversal** — (|inj| − |W_unl − W_pre|) / (|inj| + ε): fractional reversal toward pretrained
   - **DirReversal** — −(unlearn · sign(inj)) / (|inj| + ε): directional reversal, normalized
   - **ERatio** — |Δw| / (|Δw_unmask| + ε): per-parameter ratio vs UnMask baseline
   - **CrossField** — min of quantile-ranked |Δw| across all 4 PII fields (cross-field consistency)
   - **Composite** — cross-validated logistic regression on all features above (upper bound)
2. **Precision@k** — fraction of top-k changed weights that fall within the GT mask
3. **Per-layer AUC** — breakdown by transformer layer and component (attention/mlp)
4. **Per-parameter statistics** — mean/std/max delta, fraction nonzero

**Workflow:**
1. Run `scripts/precision_metrics.py` to compute metrics and save to cache
2. Open this notebook to load cached results and iterate on plots

All heavy computation (loading model weights, computing deltas) happens in the script.
This notebook only loads lightweight cached files (JSON + npz + parquet).

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from pathlib import Path

## Configuration

In [ ]:
# --- CONFIGURE THESE ---
import os
OUTPUT_DIR = os.path.join(os.environ['HOME'], 'OLMoBenchOutputs', 'saves', 'Train_95frozen_FullSubset')

FIELDS = ['Email_Address', 'Phone_Number', 'Birth_City', 'Drivers_License']
METHODS = ['SimNPO', 'MemFlex', 'AlphaEdit', 'OracleGrad']
FORGET_ONLY_METHODS = {'OracleGrad'}  # only evaluated with forget-only GT mask
UNMASK_EXCLUDED_METHODS = {'OracleGrad'}  # no UnMask counterpart

# Which mask types to plot (computed by the script)
# 'unified' = forget | retain mask, 'forget' = forget-only mask
MASK_TYPES = ['unified', 'forget']

# All metric types produced by the script
ALL_METRICS = ['raw', 'qtile', 'compnorm', 'contrast', 'contrastnorm', 'contrastln',
               'signrev', 'layernorm', 'reversal', 'dirreversal', 'eratio', 'crossfield', 'composite']
# Metrics that require UnMask counterpart (excluded for OracleGrad)
UNMASK_METRICS = {'compnorm', 'contrast', 'contrastnorm', 'contrastln', 'eratio'}

CACHE_DIR = Path(OUTPUT_DIR) / 'cached_notebook_files' / 'precision_metrics'
print(f'Cache dir: {CACHE_DIR}')
print(f'Exists: {CACHE_DIR.exists()}')

## Load cached results

Load pre-computed metrics from the cache directory created by `scripts/precision_metrics.py`.
Each experiment (field x method x mask_type) has:
- `metrics.json` — AUC values, precision@k, per-layer AUCs
- `roc_curves.npz` — downsampled FPR/TPR arrays (~1000 points, float16)
- `param_stats.parquet` — per-parameter weight delta statistics

In [ ]:
def load_experiment(mask_type, field, method):
    """Load cached results for a single (mask_type, field, method) experiment."""
    exp_dir = CACHE_DIR / mask_type / field / method
    if not exp_dir.exists():
        return None

    # Scalar metrics
    with open(exp_dir / 'metrics.json') as f:
        metrics = json.load(f)

    # Convert layer_aucs keys back from "layer_comp" strings to (int, str) tuples
    layer_aucs = {}
    for k, v in metrics.pop('layer_aucs', {}).items():
        parts = k.split('_', 1)  # e.g. "0_attention" -> (0, "attention")
        layer_aucs[(int(parts[0]), parts[1])] = v
    metrics['layer_aucs'] = layer_aucs

    # ROC curves (stored as float16 for compactness, upcast for plotting)
    curves = np.load(exp_dir / 'roc_curves.npz')
    for m in ALL_METRICS:
        fpr_key, tpr_key = f'fpr_{m}', f'tpr_{m}'
        if fpr_key in curves:
            metrics[fpr_key] = curves[fpr_key].astype(np.float32)
            metrics[tpr_key] = curves[tpr_key].astype(np.float32)
        else:
            # Metric not available for this experiment
            metrics[fpr_key] = np.array([0., 1.])
            metrics[tpr_key] = np.array([0., 1.])

    # Param stats
    metrics['param_stats'] = pd.read_parquet(exp_dir / 'param_stats.parquet')

    return metrics


# Load all experiments into nested dict: results[mask_type][field][method]
results = {}
loaded = 0
missing = 0

for mask_type in MASK_TYPES:
    results[mask_type] = {}
    for field in FIELDS:
        results[mask_type][field] = {}
        for method in METHODS:
            # Skip methods that don't apply to this mask type
            if mask_type == 'unified' and method in FORGET_ONLY_METHODS:
                continue
            r = load_experiment(mask_type, field, method)
            if r is not None:
                results[mask_type][field][method] = r
                loaded += 1
            else:
                missing += 1

print(f'Loaded {loaded} experiments, {missing} missing')
if missing > 0:
    print('Missing experiments (run scripts/precision_metrics.py first):'
          ' check CACHE_DIR above')

## Per-Component Weight Delta Statistics

Aggregated mean |Δw| by component type (attention, mlp) across fields and methods.
Shows how much each method changes different parts of the model.

In [ ]:
# Per-parameter statistics table (aggregated by component type)
# Using 'unified' mask results where available, falling back to 'forget'
for mask_type in MASK_TYPES:
    mresults = results[mask_type]
    agg_rows = []
    for field in FIELDS:
        for method in METHODS:
            if method not in mresults.get(field, {}):
                continue
            ps = mresults[field][method]['param_stats']
            for comp, group in ps.groupby('component'):
                agg_rows.append({
                    'Field': field.replace('_', ' '), 'Method': method, 'Component': comp,
                    'Mean Delta': np.average(group['mean_delta'], weights=group['numel']),
                    'Max Delta': group['max_delta'].max(),
                    'Frac Nonzero': np.average(group['frac_nonzero'], weights=group['numel']),
                    'Num Params': len(group),
                    'Total Elements': group['numel'].sum(),
                })

    if not agg_rows:
        continue
    agg_df = pd.DataFrame(agg_rows)
    mask_label = 'Forget | Retain' if mask_type == 'unified' else 'Forget Only'
    print(f'\n=== Per-component Mean Delta (GT Mask: {mask_label}) ===')
    display(agg_df.pivot_table(index=['Field', 'Method'], columns='Component',
                               values='Mean Delta', aggfunc='first'))

## AUC Summary Table

ROC AUC for each (field, method) pair across all scoring modes:
- **Raw |Δw|** — absolute weight change (global ranking)
- **Quantile** — within-component percentile rank
- **CompNorm** — delta normalized by UnMask baseline mean (requires UnMask)
- **Contrast** — |Δ_mask| − |Δ_unmask| (requires UnMask)
- **ContrastNorm** — symmetric contrast index [-1, 1] (requires UnMask)
- **ContrastLN** — layer-normalized contrast (requires UnMask)
- **SignRev** — sign-reversal of injection direction
- **LayerNorm** — |Δw| normalized by (layer, component) std
- **Reversal** — fractional reversal toward pretrained, normalized by training delta
- **DirReversal** — directional reversal, normalized by |injection|
- **ERatio** — per-parameter ratio |Δ| / |Δ_unmask| (requires UnMask)
- **CrossField** — min of quantile-ranked deltas across all PII fields
- **Composite** — logistic regression on all features (upper bound)

In [ ]:
# AUC summary tables — one per mask type
for mask_type in MASK_TYPES:
    mresults = results[mask_type]
    mask_label = 'Forget | Retain' if mask_type == 'unified' else 'Forget Only'
    rows = []
    for field in FIELDS:
        for method in METHODS:
            if method not in mresults.get(field, {}):
                continue
            r = mresults[field][method]
            row = {'Field': field.replace('_', ' '), 'Method': method}
            for m in ALL_METRICS:
                auc_val = r.get(f'auc_{m}', float('nan'))
                # Mark UnMask-dependent metrics as NaN for excluded methods
                if m in UNMASK_METRICS and method in UNMASK_EXCLUDED_METHODS:
                    auc_val = float('nan')
                row[f'AUC ({m})'] = auc_val
            rows.append(row)
    if not rows:
        continue
    prec_df = pd.DataFrame(rows)
    print(f'\nGT Mask: {mask_label}')
    display(prec_df)

## ROC Curves

Each subplot shows one PII field. Rows correspond to scoring modes.
The diagonal (dashed) is the random baseline (AUC = 0.5).
Higher AUC means the method's weight changes are better concentrated on the ground-truth
masked parameters (i.e., more precise unlearning).

Metrics requiring an UnMask counterpart (CompNorm, Contrast, ContrastNorm, ContrastLN)
are not available for OracleGrad.

In [ ]:
colors = {'AlphaEdit': '#e41a1c', 'MemFlex': '#377eb8', 'SimNPO': '#4daf4a', 'OracleGrad': '#984ea3'}

METRIC_LABELS = {
    'raw': 'Raw |Δ|', 'qtile': 'Quantile', 'compnorm': 'CompNorm',
    'contrast': 'Contrast', 'contrastnorm': 'ContrastNorm', 'contrastln': 'ContrastLN',
    'signrev': 'SignRev', 'layernorm': 'LayerNorm',
    'reversal': 'Reversal', 'dirreversal': 'DirReversal',
    'eratio': 'ERatio', 'crossfield': 'CrossField', 'composite': 'Composite',
}

for mask_type in MASK_TYPES:
    mresults = results[mask_type]
    mask_label = 'Forget | Retain' if mask_type == 'unified' else 'Forget Only'

    row_configs = [(m, f'fpr_{m}', f'tpr_{m}', f'auc_{m}') for m in ALL_METRICS]
    n_rows = len(row_configs)
    n_cols = len(FIELDS)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))

    for col, field in enumerate(FIELDS):
        for row, (metric, fpr_key, tpr_key, auc_key) in enumerate(row_configs):
            ax = axes[row][col]
            score_label = METRIC_LABELS.get(metric, metric)
            is_unmask_metric = metric in UNMASK_METRICS
            for method in METHODS:
                if method not in mresults.get(field, {}):
                    continue
                r = mresults[field][method]
                auc_val = r.get(auc_key, float('nan'))
                # Skip UnMask-dependent metrics for excluded methods
                if is_unmask_metric and method in UNMASK_EXCLUDED_METHODS:
                    continue
                if np.isnan(auc_val):
                    continue
                ax.plot(r[fpr_key], r[tpr_key],
                        label=f"{method} ({auc_val:.3f})",
                        color=colors[method], linewidth=2)
            ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
            ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
            ax.set_title(f"{field.replace('_', ' ')}\n({score_label})", fontsize=9)
            ax.legend(fontsize=7); ax.set_aspect('equal')

    fig.suptitle(f'ROC Curves: Weight Change vs Mask ({mask_label})', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

## Per-Layer AUC Heatmaps

Each cell shows the AUC for a specific (layer, component) pair. Columns alternate between
Attention (A) and MLP (M) for each transformer layer. Warmer colors = higher AUC = the method
concentrates changes more precisely on the GT mask at that layer.

In [ ]:
for mask_type in MASK_TYPES:
    mresults = results[mask_type]
    mask_label = 'Forget | Retain' if mask_type == 'unified' else 'Forget Only'

    fig, axes = plt.subplots(1, 4, figsize=(20, 6))
    layer_range = range(16)
    comps = ['attention', 'mlp']
    col_labels = [f'L{l}_{c[0].upper()}' for l in layer_range for c in comps]

    # Only include methods that have data for this mask type
    avail_methods = [m for m in METHODS if any(m in mresults.get(f, {}) for f in FIELDS)]

    for ax, field in zip(axes, FIELDS):
        matrix = []
        for method in avail_methods:
            row = []
            la = mresults.get(field, {}).get(method, {}).get('layer_aucs', {})
            for l in layer_range:
                for c in comps:
                    row.append(la.get((l, c), float('nan')))
            matrix.append(row)

        matrix = np.array(matrix)
        im = ax.imshow(matrix, aspect='auto', cmap='RdYlGn', vmin=0.3, vmax=1.0)
        ax.set_yticks(range(len(avail_methods)))
        ax.set_yticklabels(avail_methods)
        ax.set_xticks(range(0, len(col_labels), 2))
        ax.set_xticklabels([f'L{l}' for l in layer_range], rotation=90, fontsize=7)
        ax.set_title(field.replace('_', ' '))

    fig.colorbar(im, ax=axes, label='AUC', shrink=0.8)
    fig.suptitle(f'Per-Layer AUC (A=Attention, M=MLP) — {mask_label}', fontsize=14)
    plt.tight_layout()
    plt.show()

## LaTeX AUC Table

Generate a LaTeX table with AUC values for inclusion in papers. Best AUC per (field, score)
row is bolded. All scoring modes are included.

In [ ]:
method_labels = {'AlphaEdit': 'AlphaEdit', 'MemFlex': 'MemFlex', 'SimNPO': 'SimNPO', 'OracleGrad': 'OracleGrad'}

METRIC_LABELS_LATEX = {
    'raw': 'Raw', 'qtile': 'Quantile', 'compnorm': 'CompNorm',
    'contrast': 'Contrast', 'contrastnorm': 'ContrastNorm', 'contrastln': 'ContrastLN',
    'signrev': 'SignRev', 'layernorm': 'LayerNorm',
    'reversal': 'Reversal', 'dirreversal': 'DirReversal',
    'eratio': 'ERatio', 'crossfield': 'CrossField', 'composite': 'Composite',
}

for mask_type in MASK_TYPES:
    mresults = results[mask_type]
    mask_desc = 'Forget $|$ Retain' if mask_type == 'unified' else 'Forget Only'
    avail_methods = [m for m in METHODS if any(m in mresults.get(f, {}) for f in FIELDS)]

    score_types = [(METRIC_LABELS_LATEX[m], f'auc_{m}', m) for m in ALL_METRICS]

    rows = []
    for field in FIELDS:
        for score_label, auc_key, metric_key in score_types:
            is_unmask = metric_key in UNMASK_METRICS
            aucs = {}
            for m in avail_methods:
                if field not in mresults or m not in mresults[field]:
                    continue
                if is_unmask and m in UNMASK_EXCLUDED_METHODS:
                    continue
                v = mresults[field][m].get(auc_key, float('nan'))
                if not np.isnan(v):
                    aucs[m] = v
            if not aucs:
                continue
            best = max(aucs.values())
            row = [field.replace('_', ' '), score_label]
            for m in avail_methods:
                if is_unmask and m in UNMASK_EXCLUDED_METHODS:
                    row.append('---')
                    continue
                v = aucs.get(m, float('nan'))
                s = f'{v:.3f}'
                if not np.isnan(v) and abs(v - best) < 1e-9:
                    s = f'\\textbf{{{s}}}'
                row.append(s)
            rows.append(row)

    if not rows:
        continue

    cols = ['Field', 'Score'] + [method_labels[m] for m in avail_methods]
    display_df = pd.DataFrame(rows, columns=cols).set_index(['Field', 'Score'])

    suffix = '_unified' if mask_type == 'unified' else '_forget'
    latex = display_df.to_latex(
        caption=f'Unlearning Precision: ROC AUC (Weight Change vs {mask_desc} Mask).',
        label=f'tab:precision_auc{suffix}',
        column_format='ll' + 'r' * len(avail_methods),
        multirow=True,
        escape=False,
    )
    latex = latex.replace('\\begin{table}', '\\begin{table}[htbp]\n\\centering\\small', 1)
    latex = re.sub(r'\\cline\{(\d+-\d+)\}', r'\\cmidrule{\1}', latex)

    mask_label = 'Forget | Retain' if mask_type == 'unified' else 'Forget Only'
    print(f'\n=== LaTeX Table (GT Mask: {mask_label}) ===\n')
    print(latex)
    display(display_df)

## Best AUC per Method (by Field)

For each field, pick the best scoring function per method and show the resulting AUC.
One table per field, rows = methods, columns = mask types.

In [ ]:
for field in FIELDS:
    rows = []
    for method in METHODS:
        row = {'Method': method}
        for mask_type in MASK_TYPES:
            mask_label = 'Unified' if mask_type == 'unified' else 'Forget Only'
            r = results.get(mask_type, {}).get(field, {}).get(method)
            if r is None:
                row[f'Best AUC ({mask_label})'] = float('nan')
                row[f'Best Score ({mask_label})'] = '---'
                continue
            best_auc, best_metric = -1, '---'
            for m in ALL_METRICS:
                if m in UNMASK_METRICS and method in UNMASK_EXCLUDED_METHODS:
                    continue
                v = r.get(f'auc_{m}', float('nan'))
                if not np.isnan(v) and v > best_auc:
                    best_auc = v
                    best_metric = METRIC_LABELS.get(m, m)
            row[f'Best AUC ({mask_label})'] = best_auc if best_auc > 0 else float('nan')
            row[f'Best Score ({mask_label})'] = best_metric
        rows.append(row)
    df = pd.DataFrame(rows).set_index('Method')
    print(f'\n=== {field.replace("_", " ")} ===')
    display(df)

In [ ]:
# Combined LaTeX table: all fields, best AUC per method
all_rows = []
for field in FIELDS:
    for method in METHODS:
        row = {'field': field.replace('_', ' '), 'method': method}
        for mask_type in MASK_TYPES:
            mk = 'unified' if mask_type == 'unified' else 'forget'
            r = results.get(mask_type, {}).get(field, {}).get(method)
            if r is None:
                row[f'auc_{mk}'] = None
                row[f'score_{mk}'] = '---'
                continue
            best_auc, best_metric = -1, '---'
            for m in ALL_METRICS:
                if m in UNMASK_METRICS and method in UNMASK_EXCLUDED_METHODS:
                    continue
                v = r.get(f'auc_{m}', float('nan'))
                if not np.isnan(v) and v > best_auc:
                    best_auc = v
                    best_metric = METRIC_LABELS_LATEX.get(m, m)
            row[f'auc_{mk}'] = best_auc if best_auc > 0 else None
            row[f'score_{mk}'] = best_metric
        all_rows.append(row)

# Format AUC values, bolding best per (field, mask_type)
for mk in ['unified', 'forget']:
    for field in FIELDS:
        field_label = field.replace('_', ' ')
        field_rows = [r for r in all_rows if r['field'] == field_label]
        vals = [r[f'auc_{mk}'] for r in field_rows if r[f'auc_{mk}'] is not None]
        best_val = max(vals) if vals else None
        for r in field_rows:
            v = r[f'auc_{mk}']
            if v is None:
                r[f'auc_{mk}_fmt'] = '---'
            elif abs(v - best_val) < 1e-9:
                r[f'auc_{mk}_fmt'] = f'\\textbf{{{v:.3f}}}'
            else:
                r[f'auc_{mk}_fmt'] = f'{v:.3f}'

cols = pd.MultiIndex.from_tuples([
    ('Forget $|$ Retain', 'AUC'), ('Forget $|$ Retain', 'Score'),
    ('Forget Only', 'AUC'), ('Forget Only', 'Score'),
])
df = pd.DataFrame(
    [[r['auc_unified_fmt'], r['score_unified'], r['auc_forget_fmt'], r['score_forget']] for r in all_rows],
    index=pd.MultiIndex.from_arrays(
        [[r['field'] for r in all_rows], [r['method'] for r in all_rows]],
        names=['Field', 'Method']
    ),
    columns=cols,
)

latex = df.to_latex(
    caption='Best AUC per method and field (picking the best scoring function).',
    label='tab:best_auc_combined',
    column_format='ll' + 'rl' * 2,
    multicolumn=True,
    multicolumn_format='c',
    multirow=True,
    escape=False,
)
latex = latex.replace('\\begin{table}', '\\begin{table}[htbp]\n\\centering\\small', 1)
latex = re.sub(r'\\cline\{(\d+-\d+)\}', r'\\cmidrule{\1}', latex)
print(latex)
display(df)